# **Генарация датасета**

Генерируем датасет, используя функцию gen_data с лекции и параметры из условия


In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm


In [2]:
params = dict(
    x1_mean=10.0,
    x1_std=2.0,
    x2_mean=8.0,
    x2_std=1.3,
    x3_mean=12.0,
    x3_std=1.8,
    corr_12=0.0,
    corr_13=0.0,
    corr_23=0.8,
    e_mean=0.0,
    e_std=10.0,
    N=1000,
    beta0=100.0,
    beta1=1.7,
    beta2=2.3,
    beta3=-1.8
)


def gen_data(y_type, params, seed):

    # К сожалению, самый простой способ избавиться от случайности -
    # вызывать один и тот же random seed каждый раз перед запуском функции
    np.random.seed(seed)

    # Генерация датасета для задачи пропущенной переменной и мультиколлинеарности
    if y_type == 'multivariate':

        # Сгенерируем значения факторов и форму зависимости y от регрессоров
        means = [params['x1_mean'], params['x2_mean'], params['x3_mean']]

        # Вычислим наполнение матрицы ковариаций
        var_1 = params['x1_std']**2
        var_2 = params['x2_std']**2
        var_3 = params['x3_std']**2

        cov_12 = params['corr_12'] * params['x1_std'] * params['x2_std']
        cov_13 = params['corr_13'] * params['x1_std'] * params['x3_std']
        cov_23 = params['corr_23'] * params['x2_std'] * params['x3_std']

        covs = [[var_1, cov_12, cov_13],
                [cov_12, var_2, cov_23],
                [cov_13, cov_23, var_3]]

        # Сгененрируем требующуюся выборку
        X = np.random.multivariate_normal(
            mean=means,
            cov=covs,
            size=params['N']
        )

        # А также сгененируем случайную ошибку как величину из нормального распределения
        e = np.random.normal(loc=params['e_mean'], scale=params['e_std'], size=params['N'])

        # Здесь мы создаем таргет 'y' как некую функцию от x1, x2 и ошибки e
        y = params['beta0']             + params['beta1'] * X[:, 0]             + params['beta2'] * X[:, 1]             + params['beta3'] * X[:, 2]             + e

        # Для удобства сохраним вектора в pandas dataframe
        dataset = pd.DataFrame({'x1': X[:, 0], 'x2': X[:, 1], 'x3': X[:, 2], 'e': e, 'y': y})

    return(dataset)


def train_model(dataset, target, feature_names, show_results=False, pairwise=False,
                return_norm_tests=False, robust=False):

    dataset = dataset.copy()

    # Создадим матрицу фичей как фактор x1 и единичный вектор, на который будет фиттиться константа
    X = sm.add_constant(dataset[feature_names])

    # Зафитим модель на данные. y - наша целевая переменная, X - матрица факторов
    if robust == True:
        model = sm.OLS(dataset[target], X).fit(cov_type='HC0')
    else:
        model = sm.OLS(dataset[target], X).fit()

    dataset[f'{target}_hat'] = model.fittedvalues
    dataset['residuals'] = model.resid

    return(dataset, model)


Проверяем корреляционную матрицу и оценку для одной выборки

In [3]:
dataset = gen_data('multivariate', params, seed=42)
dataset, model = train_model(dataset, target='y', feature_names=['x1', 'x2', 'x3'])

sample_corr = dataset[['x1', 'x2', 'x3']].corr().round(3)
sample_summary = pd.DataFrame(
    {
        'coef': model.params,
        'std_err': model.bse,
        'ci_low': model.conf_int()[0],
        'ci_high': model.conf_int()[1],
    }
).round(4)

print('Корреляционная матрица сгенерированных факторов (seed=42):')
print(sample_corr.to_string())
print()
print('Оценки OLS и 95%-доверительные интервалы для одной выборки:')
print(sample_summary.to_string())


Корреляционная матрица сгенерированных факторов (seed=42):
       x1     x2     x3
x1  1.000  0.028  0.053
x2  0.028  1.000  0.794
x3  0.053  0.794  1.000

Оценки OLS и 95%-доверительные интервалы для одной выборки:
           coef  std_err   ci_low   ci_high
const  102.3174   2.7372  96.9461  107.6887
x1       1.5670   0.1614   1.2503    1.8838
x2       2.7335   0.4270   1.8956    3.5715
x3      -2.1864   0.3053  -2.7855   -1.5872


# **Проводим симуляцию выборки на 10000 итерациях**

In [4]:
true_betas = {
    'x1': params['beta1'],
    'x2': params['beta2'],
    'x3': params['beta3'],
}


def simulate_coverage(config, n_iter=10_000):
    records = []

    for seed in range(n_iter):
        dataset_iter = gen_data('multivariate', config, seed)
        _, model_iter = train_model(dataset_iter, target='y', feature_names=['x1', 'x2', 'x3'])
        conf_int = model_iter.conf_int(alpha=0.05)

        for feature, true_beta in true_betas.items():
            low, high = conf_int.loc[feature]
            records.append(
                {
                    'feature': feature,
                    'covered': int(low <= true_beta <= high),
                    'coef': model_iter.params[feature],
                    'ci_width': high - low,
                }
            )

    return pd.DataFrame(records)


In [5]:
simulation_results = simulate_coverage(params, n_iter=10_000)

coverage_summary = (
    simulation_results.groupby('feature')
    .agg(
        coverage=('covered', 'mean'),
        mean_coef=('coef', 'mean'),
        std_coef=('coef', 'std'),
        mean_ci_width=('ci_width', 'mean'),
    )
    .round(4)
)

print('Итог по 10 000 итерациям:')
print(coverage_summary.to_string())
print()
print(f"Доля покрытий для beta1: {coverage_summary.loc['x1', 'coverage']:.4f}")
print(f"Доля покрытий для beta2: {coverage_summary.loc['x2', 'coverage']:.4f}")
print(f"Доля покрытий для beta3: {coverage_summary.loc['x3', 'coverage']:.4f}")


Итог по 10 000 итерациям:
         coverage  mean_coef  std_coef  mean_ci_width
feature                                              
x1         0.9524     1.7027    0.1570         0.6216
x2         0.9467     2.2978    0.4115         1.5943
x3         0.9476    -1.7977    0.2951         1.1512

Доля покрытий для beta1: 0.9524
Доля покрытий для beta2: 0.9467
Доля покрытий для beta3: 0.9476


## **Вывод**
Результат симуляции на 10 000 итераций:
- `beta1`: `0.9524`
- `beta2`: `0.9467`
- `beta3`: `0.9476`

Утверждение про `beta1` подтверждается: покрытие очень близко к 95%.
Для `beta2` и `beta3` покрытие тоже близко к 95%, но интервалы шире из-за сильной корреляции между `x2` и `x3`.
